<a href="https://colab.research.google.com/github/shikhar286/Agentic-AI-Portfolio-Shikhar-2026/blob/main/MCP_SelectBudgetFitHotelFromHotelList.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


**Problem Statement**

Organizations often need AI assistants to access real-time business data such as hotel listings, pricing, availability, customer records, or internal documents. Traditional chatbot implementations usually rely on static prompts or hardcoded sample data, which prevents the assistant from giving accurate, live, and context-aware responses.

The goal of this project is to build a **real-time AI assistant using MCP (Model Context Protocol)** that allows an LLM such as Claude to securely discover and call external tools. These tools will connect to live data sources such as APIs, databases, or internal systems.

In this use case, the assistant should help users search hotel or apartment listings based on filters such as location, price, bedrooms, and listing details. Instead of storing data inside the chatbot, the MCP server will expose reusable tools like `search_listings` and `get_listing_details`, which fetch real-time information from the backend system.

This approach makes the AI assistant more dynamic, scalable, and production-ready by separating the LLM reasoning layer from the business logic and data access layer.


In [ ]:
!pip install -q fastmcp anthropic

import json
import os
from anthropic import Anthropic
from fastmcp import FastMCP
from fastmcp.client import Client

# ---------------------------------------
# 1. Create Anthropic Client
# ---------------------------------------
# Recommended: Store your API key in environment variable / Colab secret
anthropic_client = Anthropic(
    api_key=input('Enter Anthropic API Key: ')
)

# ---------------------------------------
# 2. Create MCP Server
# ---------------------------------------
mcp_server = FastMCP("Enterprise Hotel Search")

# ---------------------------------------
# 3. Sample Hotel Data
# ---------------------------------------
LISTINGS = [
    {"id": "1", "title": "Business Studio", "price": 85, "location": "New York", "bedrooms": 1},
    {"id": "2", "title": "Executive Loft", "price": 150, "location": "Brooklyn", "bedrooms": 2},
    {"id": "3", "title": "Team Stay Apartment", "price": 120, "location": "Queens", "bedrooms": 2},
    {"id": "4", "title": "Budget Studio", "price": 95, "location": "New York", "bedrooms": 1}
]

# ---------------------------------------
# 4. MCP Tool 1: Search Listings
# ---------------------------------------
@mcp_server.tool()
def search_listings(max_price: int = 1000, location: str = "") -> str:
    """Search hotel listings by price and location."""

    results = LISTINGS

    if max_price:
        results = [
            hotel for hotel in results
            if hotel["price"] <= max_price
        ]

    if location:
        results = [
            hotel for hotel in results
            if location.lower() in hotel["location"].lower()
        ]

    return json.dumps(results, indent=2)


# ---------------------------------------
# 5. MCP Tool 2: Get Listing Details
# ---------------------------------------
@mcp_server.tool()
def get_listing_details(listing_id: str) -> str:
    """Get full details of a hotel listing using listing ID."""

    for hotel in LISTINGS:
        if hotel["id"] == listing_id:
            hotel_details = {
                **hotel,
                "amenities": ["WiFi", "Workspace", "Breakfast"],
                "rating": 4.8
            }
            return json.dumps(hotel_details, indent=2)

    return json.dumps({"error": "Listing not found"}, indent=2)


# ---------------------------------------
# 6. Run AI Agent with MCP Tools
# ---------------------------------------
async def run_agent(user_query: str):

    """
    mcp_client is MCP Client object. Once that object is active inside your "async with" block you can use method like:
    await mcp_client.list_tools() (to fetch available capability schemas)
    await mcp_client.call_tool(...) (to execute a specific remote server operation)
    """
    async with Client(mcp_server) as mcp_client:


        # Get tools from MCP server.
        '''
        The await keyword means the program will pause execution at this exact line and wait for the MCP Server
        to respond with the tool schemas before moving down to the next line of code.
        Even though it allows other tasks to run globally, the local code inside this specific function cannot move to the next line
        '''
        mcp_tools = await mcp_client.list_tools()

        '''
        It translates raw mcp server outputs into standard configurations an LLM understands by formatting the properties
        into claude_tools. When mcp server gives list of tools to MCPClient, it passes it to MCPHost. Host will convert this list of
        tools to LLM format. Finally anthropic_client will put this list of tools in anthropic format on cloud and give it to anthropic
        our LLM here.
        '''
        claude_tools = [
            {
                "name": tool.name,
                "description": tool.description,
                "input_schema": tool.inputSchema,
                #It uses a standard format called JSON Schema to tell the LLM, "If you want to use this tool,
                #your arguments must look exactly like this."

            }
            for tool in mcp_tools
        ]

        # Initializing the memory with the user's question

        messages = [
            {"role": "user", "content": user_query}
        ]

        while True:

            '''
            you hand the user's query (and any tool history) to the LLM so it can decide what to do next.
            '''
            response = anthropic_client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=1000,
                tools=claude_tools, #this has list of tools available with mcp server
                messages=messages   #user's query (and any tool history)
            )

            # If Claude gives final answer, print it
            if response.stop_reason != "tool_use":
                '''
                "tool_use": Claude realizes it cannot answer the user on its own and requires data from an external tool.
                "end_turn": Claude has finished its thoughts and is providing its final conversational response to the user.
                '''
                print(response.content[0].text)
                return

            # When Claude responds, it doesn't just return a plain string of text like "Hello!". Instead, it sends back a complex Python object containing a list (an array) called content.
            #This line,is just grabbing the very first item out of that list because that's where Claude puts its actual answer=> tool to call.
            tool_request = response.content[0]
            '''
            currently claude respose has:
            response.content = [
            {                                    ← this is index [0]
              "type":  "tool_use",
              "id":    "tool_abc_123",         ← unique ID for this tool call
              "name":  "search_listings",      ← tool_request.name
              "input": {                       ← tool_request.input
            "location":  "New York",
            "max_price": 100
                        }
            }
            ]
            '''



            '''
            tool_request.name   →  "search_listings"
            tool_request.input  →  {"location": "New York", "max_price": 100}
            tool_request.id     →  "tool_abc_123"   ← used later to match result back
            '''
            # Run the requested MCP tool
            tool_result = await mcp_client.call_tool(
                tool_request.name,
                arguments=tool_request.input
            )
            '''
            mcp_client.call_tool(
            "search_listings",                           ← which tool to run
            arguments={"location":"New York","max_price":100}  ← with what inputs)
            '''


            # Saving Claude's tool request to memory
            messages.append({
                "role": "assistant",
                "content": response.content
            })


            '''
            tool_result.content[0].text = '[
            {"id":"1", "title":"Business Studio", "price":85,  "location":"New York"},
            {"id":"4", "title":"Budget Studio",   "price":95,  "location":"New York"}
            ]'
            '''
            # Saving the tool's execution result to memory
            messages.append({
                "role": "user",
                "content": [
                    {
                        "type": "tool_result",
                        "tool_use_id": tool_request.id,
                        "content": tool_result.content[0].text
                    }
                ]
            })

            '''
            messages = [

    # ── entry 1 (added before loop started) ──────────────────────
    {
        "role": "user",
        "content": "Find me affordable apartments under $100 in New York with rating of 4.8"
    },

    # ── entry 2 (added at end of Loop 1) ─────────────────────────
    {
        "role": "assistant",
        "content": [
            {
                "type":  "tool_use",
                "id":    "tool_abc_123",
                "name":  "search_listings",
                "input": {"location": "New York", "max_price": 100}
            }
        ]
    },

    # ── entry 3 (added at end of Loop 1) ─────────────────────────
    {
        "role": "user",
        "content": [
            {
                "type":        "tool_result",
                "tool_use_id": "tool_abc_123",
                "content":     '[{"id":"1","title":"Business Studio","price":85,"location":"New York"},
                                 {"id":"4","title":"Budget Studio","price":95,"location":"New York"}]'
            }
        ]
    }

]
            '''

In [10]:
await run_agent("Fund me a hotel in New York within budget of $100")

Great! I found 2 hotels in New York within your $100 budget:

1. **Business Studio** - $85/night
   - Location: New York
   - Bedrooms: 1
   - Listing ID: 1

2. **Budget Studio** - $95/night
   - Location: New York
   - Bedrooms: 1
   - Listing ID: 4

Would you like me to get more detailed information about either of these listings? Just let me know which one interests you!
